In [25]:
! pip install peft bitsandbytes accelerate

In [26]:
import os
from typing import List, Dict, Any

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, BitsAndBytesConfig, AutoConfig, set_seed
from torch.utils.data import Dataset
from datasets import load_dataset
from peft import PeftModel, get_peft_model, LoraConfig

set_seed(12, True)

os.environ["WANDB_DISABLED"] = "true"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

# Gradient Accumulation

Давайте реализуем собственную аккумуляцию градиентов.
Ниже описано обучение обычного линейного слоя. Клеткой ниже этот код скопирован, там необходимо написать аккумуляцию ргадиентов.

In [27]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
input_size = 512
output_size = 256
batch_size = 64
gradient_accumulation_steps = 4



model = nn.Linear(input_size, output_size).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.001)

x = torch.randn(batch_size, input_size).to(device)
y = torch.randn(batch_size, output_size).to(device)
loss_fn = nn.MSELoss()
for i in range(1000):
    optimizer.zero_grad()
    output = model(x)
    loss = loss_fn(output, y)
    loss.backward()
    optimizer.step()

print(loss.item())

1.1878372430801392


Число шагов в аккумуляции определяется параметром gradient_accumulation_steps - это число шагов, которое мы хотим сделать перед оптимизацией.
Вам нужно поправить цикл обучения следующим образом:
1. Разбить текущий батч на gradient_accumulation_steps частей
2. Пройтись по каждому подбатчу (микробатчу), посчитать на нем функцию потерь, посчитать градиенты. Подумайте, нужно ли на что-либо делить или умножать функцию потерь, чтобы сохранился тот же масштаб обучения?
3. После прохождения всех микробатчей нужно сделать шаг оптимизации

In [28]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
input_size = 512
output_size = 256
batch_size = 64
gradient_accumulation_steps = 4


set_seed(12, True)
model = nn.Linear(input_size, output_size).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.001)

x = torch.randn(batch_size, input_size).to(device)
y = torch.randn(batch_size, output_size).to(device)
loss_fn = nn.MSELoss()
for i in range(1000):
    optimizer.zero_grad()
    losses = 0

    for substep in range(gradient_accumulation_steps):
        # Вычисляем размер микро-батча
        micro_batch_size = batch_size // gradient_accumulation_steps
        xx = x[substep * micro_batch_size: (substep + 1) * micro_batch_size]
        yy = y[substep * micro_batch_size: (substep + 1) * micro_batch_size]

        output = model(xx)
        loss = loss_fn(output, yy)
        # Делим лосс на количество шагов аккумуляции для сохранения масштаба градиента
        (loss / gradient_accumulation_steps).backward()
        losses += (loss / gradient_accumulation_steps).item()
    # Делаем шаг оптимизации после всех микро-батчей
    optimizer.step()

print(loss.item())

1.2108761072158813


# QLORA
Необходимо использовать аккумуляцию градиентов, чекпоинтинг активаций и обучение qlora.

In [29]:
model_name = "NousResearch/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [30]:
imdb = load_dataset("imdb")

Наша задача научиться генерировать класс текста posive или negative, чтобы сэкономить на fewshot промпте.

Давайте напишем collate_fn, которая собирает сэмпл следующим образом:

если текст имеет метку 1
`{text} ||| posive eos`
или
`{text} ||| negatve eos`
если текст имеет метку 0. (в качестве eos можно использовать tokenizer.eos_token_id)

Символы ||| нужны нам, чтобы разделить входной текст и метку, иначе модель может не понять, что нужно генерировать метку и продолжит генерировать текст. Таким образом мы научим модель после ||| генерировать положительный или отрицательнй отзыв стоит до этого.


Возвращать нужно словарь из 3х элементов:
1. input_ids - LongTensor токенов. В качестве паддинга нужно использовать tokenizer.eos_token_id.
2. attention_mask - LongTensor той же размерности, что и input_ids. 0 там, где стоят паддинги, 1 в остальных позициях
3. labels - метки, которые мы предсказыаем. Должен быть равен -100 на всех позициях, кроме позиций, которые соответствуют метке и eos символу.
Например
```python
tokenizer.encode("some text ||| positive </s>") # [1, 777, 1426, 3830, 29989, 6374, 2]
labels = [-100, -100, -100, -100, -100, 6374, 2]
```

Т.е. метки должны быть -100, кроме позиций, соответствующих предсказываемым токенам.

In [31]:
def collate_fn(batch: List[Dict[str, Any]]):
    class_mapping = {0: "negative", 1: "positive"}
    # Ограничиваем текст и добавляем разделитель
    texts = [sample["text"][:512] + " ||| " for sample in batch]

    # Токенизируем входную часть
    token_ids = [tokenizer.encode(t) for t in texts]
    len_pre = [len(ids) for ids in token_ids]

    # Добавляем целевую метку и EOS токен
    for ids, sample in zip(token_ids, batch):
        label = sample["label"]
        ids += tokenizer.encode(class_mapping[label], add_special_tokens=False) + [tokenizer.eos_token_id]

    # Паддинг до максимальной длины в батче
    input_ids = pad_sequence([torch.LongTensor(ids) for ids in token_ids], True, tokenizer.eos_token_id)
    maxlength = input_ids.shape[1]

    # Attention mask: 1 для реальных токенов, 0 для паддинга
    attention_mask = torch.LongTensor([[1] * len_pre[i] + [0] * (maxlength - len_pre[i]) for i in range(len(batch))])

    # Labels: -100 для входного текста, ID токенов для метки + EOS
    labels = []
    for i in range(len(batch)):
        sample_labels = [-100] * len_pre[i]
        # Добавляем ID токенов метки (они идут после len_pre)
        sample_labels.append(input_ids[i][len_pre[i]])
        sample_labels.append(input_ids[i][len_pre[i] + 1])
        sample_labels += [-100] * (maxlength - len(sample_labels))
        labels.append(sample_labels)

    return {
         "input_ids": input_ids,
         "attention_mask": attention_mask,
         "labels": torch.LongTensor(labels)
    }

res = collate_fn([imdb["train"][0], imdb["train"][12505], imdb["train"][2]])

# Для проверки с </s> нужно использовать skip_special_tokens=False
assert tokenizer.decode(res["input_ids"][res["labels"] != -100], skip_special_tokens=False) == "negative</s> positive</s> negative</s>"

Далее нам нужно создать модель в nf4, т.е. 4-битной квантизации. Конфиг уже написан, нужно лишь подать его в модель. После этого нужно:
1. Создать конфиг адаптера LoraConfig (используйте r=8 или r=4, если будет OOM) и создать модель
2. Создать модель с адаптером с помощью PeftModel и LoraConfig
3. Чтобы обучение шло только по lora частям, нужно пройтись по всем параметрам модели с помощью model.named_parameters() и проставить у параметров, соответствующих lora атрибут requires_grad = True, а у всех остальных False

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
)

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=4,  # Можно уменьшить до 8 или 4 при OOM
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = PeftModel(model, peft_config)

# Замораживаем базовую модель, оставляем обучаемыми только веса LoRA
for name, p in model.named_parameters():
    if "lora" in name:
        p.requires_grad = True
    else:
        p.requires_grad = False

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Осталось самое важное, аргументы обучения. Обязательно заполните следующие параметры:

1. Батч сайз и число шагов аккумуляции выставьте так, чтобы эффективный батч сайз был 16
2. Включите чекпоинтинг активаций

In [ ]:
args = TrainingArguments(
    output_dir="ckpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Эффективный батч = 1 * 16 = 16
    learning_rate=1e-4,
    max_steps=20,
    logging_strategy="steps",
    logging_steps=1,
    bf16=True,
    report_to=None,
    remove_unused_columns=False,
    gradient_checkpointing=True  # Обязательно для экономии памяти
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=imdb["train"],
    tokenizer=tokenizer,
    data_collator=collate_fn,
)
trainer.train()

Давайте протестируем, что модель что-то выучила

In [ ]:
input_text = imdb["test"][0]["text"] + " ||| "
label = imdb["test"][0]["label"]
x = tokenizer(input_text, return_tensors="pt")
for k, v in x.items():
    x[k] = v.cuda()

print(label)
g = model.generate(**x, max_new_tokens=2, do_sample=False)
print(tokenizer.decode(g[0].tolist()))

Использование технологий QLoRA (квантование до 4 бит) и LoRA (обучение только адаптеров) делает возможным дообучение моделей уровня 7B (как Llama-2-7b) на потребительском железе (например, GPU с 16 ГБ видеопамяти). Без этих технологий потребовалось бы несколько десятков гигабайт VRAM.
Вывод: Для успешного обучения на ограниченных ресурсах необходимо комбинировать несколько техник оптимизации:
1.Gradient Checkpointing: Обязательно включать в TrainingArguments (gradient_checkpointing=True). Это жертвует скоростью ради значительной экономии памяти при вычислении градиентов.
2.Gradient Accumulation: Позволяет имитировать большой батч (например, 16), собирая градиенты от нескольких микро-батчей (например, 16 шагов по 1 сэмплу), не увеличивая пиковое потребление памяти.
3.Квантование: Использование nf4 и double_quant в BitsAndBytesConfig.